# P1-2 · Đánh giá model đúng cách

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
X_train, y_train = make_classification(n_samples=600, n_features=20, n_informative=5, weights=[0.7], random_state=0)
model = LogisticRegression(max_iter=1000)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
print(f"F1 = {scores.mean():.3f} ± {scores.std():.3f}")   # báo cáo cả độ lệch giữa các fold, không chỉ trung bình

F1 = 0.799 ± 0.037


In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 5000))           # 5.000 đặc trưng nhiễu thuần
y = rng.integers(0, 2, size=100)           # nhãn tung đồng xu

# SAI: chọn 20 đặc trưng "tốt nhất" trên toàn bộ dữ liệu, rồi mới cross-validate
Xs = SelectKBest(f_classif, k=20).fit_transform(X, y)
cross_val_score(LogisticRegression(), Xs, y, cv=cv).mean()        # 0.87 (!)

# ĐÚNG: việc chọn đặc trưng nằm trong pipeline, chạy lại trên từng fold train
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
cross_val_score(pipe, X, y, cv=cv).mean()                          # 0.50

0.8699999999999999
0.5


## Bài tập 3: Tìm rò rỉ {khái niệm}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# (a)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y)

# (b) dự đoán giá cổ phiếu ngày mai
Xtr, Xte, ytr, yte = train_test_split(X_by_day, y_by_day, shuffle=True)

# (c) dự đoán khách có rời bỏ dịch vụ không
features = ["tuổi", "số đơn 6 tháng", "ngày hủy tài khoản"]

# (d) phân loại ảnh X-quang, mỗi bệnh nhân có 5 ảnh
Xtr, Xte = train_test_split(all_images, test_size=0.2)

## Bài tập 4: Chọn ngưỡng tối ưu F1 {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# dữ liệu mất cân bằng để thử: 10% lớp dương
X, y = make_classification(n_samples=3000, n_features=20, n_informative=6, weights=[0.9], flip_y=0.03, random_state=1)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)

def best_threshold(y_val, proba_val):
    ths = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(y_val, proba_val >= t) for t in ths]
    i = int(np.argmax(scores))
    return ths[i], scores[i]

t, f = best_threshold(y_val, model.predict_proba(X_val)[:, 1])
p_test = model.predict_proba(X_test)[:, 1]
print(f"ngưỡng {t:.2f}: F1 test {f1_score(y_test, p_test >= t):.3f} | ngưỡng 0.5: {f1_score(y_test, p_test >= 0.5):.3f}")

ngưỡng 0.25: F1 test 0.681 | ngưỡng 0.5: 0.579


## Bài tập 5: Dự báo doanh số: chia ngẫu nhiên hay theo thời gian {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
t = np.arange(730)
sales = 100 + 0.15 * t + 20 * np.sin(2 * np.pi * t / 365) + 8 * np.sin(2 * np.pi * t / 7) + rng.normal(0, 5, 730)
X = np.column_stack([sales[i:i + 7] for i in range(723)]).T   # mỗi hàng: 7 ngày trước
y = sales[7:]                                                   # ngày cần dự báo
# TODO: mae_ngau_nhien, mae_thoi_gian (RandomForestRegressor(n_estimators=100, random_state=0))

In [ ]:
def danh_gia(X_tr, X_te, y_tr, y_te):
    m = RandomForestRegressor(n_estimators=100, random_state=0).fit(X_tr, y_tr)
    return mean_absolute_error(y_te, m.predict(X_te))

mae_ngau_nhien = danh_gia(*train_test_split(X, y, test_size=0.2, random_state=0))
cut = int(0.8 * len(y))
mae_thoi_gian = danh_gia(X[:cut], X[cut:], y[:cut], y[cut:])
print(f"chia ngẫu nhiên: MAE {mae_ngau_nhien:.1f}   chia theo thời gian: MAE {mae_thoi_gian:.1f}")

chia ngẫu nhiên: MAE 5.4   chia theo thời gian: MAE 7.2


## Bài tập 6: Chọn độ phức tạp bằng validation {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
rng = np.random.default_rng(2)
temp = rng.uniform(10, 40, 60)
power = 50 + 0.15 * (temp - 24) ** 2 + rng.normal(0, 6, 60)
x_tr, y_tr = temp[:20, None], power[:20]          # 20 ngày để train
x_va, y_va = temp[40:, None], power[40:]          # 20 ngày validation; 20 ngày giữa để dành làm test
# TODO: với bậc d = 1..15: fit, rmse train/val; bac_tot

In [ ]:
bac = range(1, 16)
rmse_tr, rmse_va = [], []
for d in bac:
    m = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(x_tr, y_tr)
    rmse_tr.append(np.sqrt(mean_squared_error(y_tr, m.predict(x_tr))))
    rmse_va.append(np.sqrt(mean_squared_error(y_va, m.predict(x_va))))
bac_tot = int(np.argmin(rmse_va) + 1)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(bac, rmse_tr, marker="o", label="train"); ax.plot(bac, rmse_va, marker="o", label="validation")
ax.set_yscale("log"); ax.axvline(bac_tot, color="gray", ls="--")
ax.set(xlabel="bậc đa thức", ylabel="RMSE (thang log)", title=f"Bậc {bac_tot} tốt nhất; bậc cao thì overfit"); ax.legend()
print("bậc tốt nhất:", bac_tot)

bậc tốt nhất: 2


## Bài tập 7: Phát hiện gian lận: khi accuracy đánh lừa {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
X, y = make_classification(n_samples=20000, n_features=12, n_informative=5, weights=[0.98], flip_y=0.005, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
# TODO: kq[tên] = (accuracy, precision, recall, f1) cho 3 model

In [ ]:
models = {"luôn không": DummyClassifier(strategy="most_frequent"),
          "logistic": LogisticRegression(max_iter=1000),
          "balanced": LogisticRegression(max_iter=1000, class_weight="balanced")}
kq = {}
for ten, m in models.items():
    p = m.fit(X_tr, y_tr).predict(X_te)
    kq[ten] = (accuracy_score(y_te, p), precision_score(y_te, p, zero_division=0), recall_score(y_te, p), f1_score(y_te, p))
    print(f"{ten:11s} acc {kq[ten][0]:.3f}  precision {kq[ten][1]:.2f}  recall {kq[ten][2]:.2f}  F1 {kq[ten][3]:.2f}")
print("PR-AUC logistic:", round(average_precision_score(y_te, models["logistic"].predict_proba(X_te)[:, 1]), 3))

luôn không  acc 0.977  precision 0.00  recall 0.00  F1 0.00
logistic    acc 0.989  precision 0.94  recall 0.55  F1 0.69
balanced    acc 0.890  precision 0.15  recall 0.85  F1 0.26
PR-AUC logistic: 0.719


## Bài tập 8: Chọn ngưỡng theo chi phí thật {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
X, y = make_classification(n_samples=20000, n_features=12, n_informative=5, weights=[0.98], flip_y=0.005, random_state=0)
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.3, stratify=y_tmp, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
CHI_PHI_FN, CHI_PHI_FP = 2000, 100         # nghìn đồng

def chi_phi(y_true, prob, nguong):
    pred = prob >= nguong
    fn = ((y_true == 1) & ~pred).sum()
    fp = ((y_true == 0) & pred).sum()
    return fn * CHI_PHI_FN + fp * CHI_PHI_FP
# TODO: nguong_tot (theo validation), cp_test_tot, cp_test_05

In [ ]:
ths = np.linspace(0.01, 0.99, 99)
p_va = model.predict_proba(X_va)[:, 1]
nguong_tot = ths[np.argmin([chi_phi(y_va, p_va, t) for t in ths])]
p_te = model.predict_proba(X_te)[:, 1]
cp_test_tot, cp_test_05 = chi_phi(y_te, p_te, nguong_tot), chi_phi(y_te, p_te, 0.5)
print(f"ngưỡng tốt nhất (validation) = {nguong_tot:.2f}")
print(f"chi phí test: {cp_test_tot / 1000:.0f} triệu ở ngưỡng đó, {cp_test_05 / 1000:.0f} triệu ở ngưỡng 0,5")

ngưỡng tốt nhất (validation) = 0.08
chi phí test: 82 triệu ở ngưỡng đó, 124 triệu ở ngưỡng 0,5
